# Investigating BCO Temperatures
Since I have time series for the Barbados GAIA and Barbados CIMH stations, I can do a statistical analysis to compare the data from the BCO with these stations.

# Imports


In [ ]:
# Imports
from datetime import datetime, timedelta
import intake
from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

# Preparing Datasets
## BCO Temperature Data
Below is the starting code from [Tropical Cloud Observations](https://tcodata.mpimet.mpg.de/intro.html) group.

In [ ]:
cat = intake.open_catalog("https://tcodata.mpimet.mpg.de/catalog.yaml")

wxt = cat.BCO.surfacemet_wxt_v1.to_dask()


In [ ]:
wxt

In [ ]:
# Time shift by 4 hours to align with the other dataset
# wxt["VEL"] = wxt["VEL"].shift(time=4, fill_value=0)
wxt["VEL"] = wxt["VEL"].assign_coords(time=wxt["VEL"].time - np.timedelta64(4, "h"))

hourly_cycle = (
    wxt["VEL"]
    .sel(time=slice("2010", "2020"))
    .groupby("time.hour")
    .mean()
)

hourly_cycle.plot()

### Absolute daily temperature
The daily Tmax, Tmin, and Tmean for the dataset is computed for all time stamps in each day.

In [ ]:
#  Daily absolute statistics
bco_abs_daily_tmax = wxt["T"].resample(time="1D").max()
bco_abs_daily_tmin = wxt["T"].resample(time="1D").min()
bco_abs_daily_tmean = wxt["T"].resample(time="1D").mean()

In [ ]:
bco_abs_daily_tmax

We note that the same result can be attained using the `groupby` function as follows:

In [ ]:
wxt["T"].groupby("time.date").max(dim="time").plot(figsize=(12, 4));

In [ ]:
bco_abs_daily_tmax = wxt["T"].groupby("time.date").max(dim="time")
bco_abs_daily_tmin = wxt["T"].groupby("time.date").min(dim="time")
bco_abs_daily_tmean = wxt["T"].groupby("time.date").mean(dim="time")

## Mean Windspeed Data
We will use the mean wind speed data from the `VEL` variable from the dataset. The variable yields $10\,\mathrm{s}$ mean wind speed. We will first resample it into $10\,\mathrm{min}$ windspeed and then into daily windspeed.

In [ ]:
minute_windspeed = wxt["VEL"].sel.resample(time="1T").mean()
minute_windspeed.plot(figsize=(12, 4));

# daily_windspeed = wxt["VEL"].resample(time="1D").mean()
# daily_windspeed.plot(figsize=(12, 4));


## Mean Wind Direction Data
We will use the mean wind direction data from the `DIR` variable from the dataset. The variable yields $10\,\mathrm{s}$ mean wind direction. We will first resample it into daily windspeed.

In [ ]:
daily_wind_direction = wxt["DIR"].resample(time="1D").mean()
daily_wind_direction.plot(figsize=(12, 4));

## Total Daily Rainfall Data
We take the sum of the rainfall data over the period so that we can compare it with the other stations on the island. Since rainfall for the previous day is recorded at 8:00 AM daily, we will shift each timestamp of the BCO data by $-12\,\mathrm{hours}$. After this, the data was resampled by the sum.

In [ ]:
# Shift time by -4 hours to convert UTC to local time and then by -8 hours to match the time
# that rainfall is recorded (locally at 8 am for the previous day)
wxt_local = wxt['R'].assign_coords(time=wxt['time'] - np.timedelta64(12, 'h'))

In [ ]:
daily_rainfall = wxt_local.resample(time='1D').sum()
daily_rainfall.plot(figsize=(12,4), label='Daily Rainfall')

In [ ]:
daily_rainfall = wxt_local.resample(time='1D').sum()

In [ ]:
# Select maximum day
max_rainfall_day = daily_rainfall.argmax(dim='time').values

In [ ]:
max_rainfall_day

In [ ]:
daily_wind_direction.to_series().index.has_duplicates


## Generating CSV
To make things easier, we will convert this data into a csv file.

In [ ]:
(pd.concat(
    [bco_abs_daily_tmax.to_series(), bco_abs_daily_tmin.to_series(), 
    bco_abs_daily_tmean.to_series(), daily_windspeed.to_series(), 
    daily_wind_direction.to_series(), daily_rainfall.to_series()], 
    axis=1, keys=['tmax','tmin','tmean', 'vmean', 'dir_mean', 'total_rainfall']
)
.reindex(pd.date_range(start="2010-12-16", end="2024-12-31", freq='1D'))
.rename_axis('Date')
.round(1)
.to_csv('2010-2024 - BCO Daily Statistics.csv')
)

## Hourly daily temperatures
The statistics are computed for the top of the hour to match the method utilized for the station datasets.

In [ ]:
time = wxt["time"].to_index()

# select times closest to top of each hour
time_hourly = time[time.minute == 0]
T_hourly = wxt["T"].sel(time=time_hourly, method="nearest")

In [ ]:
bco_daily_tmax = T_hourly.resample(time="1D").max()
bco_daily_tmin = T_hourly.resample(time="1D").min()
# bco_daily_tmean = T_hourly.resample(time="1D").mean()

# Loading Data
## Load GAIA and CIMH station data

In [ ]:
# Load observations
tmax_obs_path = Path("1985-2024 - Daily Tmax (matching stations).csv")
tmin_obs_path = Path("1985-2024 - Daily Tmin (matching stations).csv")
rain_obs_path = Path("1985-2024 - Daily Rainfall.csv")

tmax_obs = pd.read_csv(
    tmax_obs_path,
    skiprows=[1, 2, 3],
    usecols=['Date', 'Barbados_GAIA', 'Barbados_CIMH'],
    parse_dates=["Date"], 
    date_format="%d-%m-%y",
    dtype="float64",
    na_values=["///"]
).set_index("Date")

tmin_obs = pd.read_csv(
    tmin_obs_path,
    skiprows=[1, 2, 3],
    usecols=['Date', 'Barbados_GAIA', 'Barbados_CIMH'],
    parse_dates=["Date"], 
    date_format="%d-%m-%y",
    dtype="float64",
    na_values=["///"]
).set_index("Date")

rain_obs = pd.read_csv(
    rain_obs_path,
    skiprows=[1, 2, 3],
    usecols=['Date', 'Barbados_GAIA', 'Barbados_CIMH'],
    parse_dates=["Date"], 
    date_format="%d-%m-%y",
    dtype="float64",
    na_values=["///"]
).set_index("Date")

rain_obs

## Load BCO Data

In [ ]:
bco_obs_path = Path("2010-2024 - BCO Daily Statistics.csv")

bco_obs = pd.read_csv(
    bco_obs_path, 
	parse_dates=["Date"], 
	dayfirst=True 			# Ensure that index is in datetime format.
).set_index("Date")

print(bco_obs)


# Plots
Now that the data is loaded, we can now create some case plots.

## Case 1: July 7 to July 15, 2023 heat wave

In [ ]:
# Settings for plotting
start_date = "2023-07-07"
end_date = "2023-07-15"

start_dt = datetime.strptime(start_date, "%Y-%m-%d")
end_dt = datetime.strptime(end_date, "%Y-%m-%d")
month_str = start_dt.strftime("%B")

unit = r"$^\circ C$"

# Create figure and axes
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, dpi=300)
plt.subplots_adjust(hspace=0.25)

# Tmax on ax1:
ax1.plot(
    bco_obs.loc[start_date:end_date].index,
    bco_obs.loc[start_date:end_date]["tmax"],
    label="BCO Tmax",
    color="#1f77b4",
    linewidth=2
)
# ax1.plot(
#     bco_abs_daily_tmax.sel(time=slice(start_date, end_date)).time,
#     bco_abs_daily_tmax.sel(time=slice(start_date, end_date)),
#     label="BCO Tmax resampled",
#     color="#1f77b4", linestyle=":",
#     linewidth=2
# )
ax1.plot(
    tmax_obs.loc[start_date:end_date].index,
    tmax_obs.loc[start_date:end_date]["Barbados_GAIA"],
    label="GAIA Tmax",
    color="#ff7f0e",
    linewidth=2
)
ax1.plot(
    tmax_obs.loc[start_date:end_date].index,
    tmax_obs.loc[start_date:end_date]["Barbados_CIMH"],
    label="CIMH Tmax",
    color="#ee0505",
    linewidth=2
)

ax1.set_title(f"Daily Maximum Temperature — Barbados ({month_str} {start_dt.day}–{end_dt.day}, {start_dt.year})",
              fontsize=14, weight="bold", pad=12)
ax1.set_ylabel(f"Tmax / {unit}", fontsize=12)

# Tmin on ax2:
ax2.plot(
    bco_obs.loc[start_date:end_date].index,
    bco_obs.loc[start_date:end_date]["tmin"],
    label="BCO Tmin",
    color="#1f77b4",
    linewidth=2
)
# ax2.plot(
#     bco_abs_daily_tmin.sel(time=slice(start_date, end_date)).time,
#     bco_abs_daily_tmin.sel(time=slice(start_date, end_date)),
#     label="BCO Tmin resampled",
#     color="#1f77b4",
#     linewidth=2, linestyle=":"
# )
ax2.plot(
    tmin_obs.loc[start_date:end_date].index,
    tmin_obs.loc[start_date:end_date]["Barbados_GAIA"],
    label="GAIA Tmin",
    color="#ff7f0e",
    linewidth=2
)
ax2.plot(
    tmin_obs.loc[start_date:end_date].index,
    tmin_obs.loc[start_date:end_date]["Barbados_CIMH"],
    label="CIMH Tmin",
    color="#ee0505",
    linewidth=2
)

ax2.set_ylabel(f"Tmin / {unit}", fontsize=12)
ax2.set_xlabel("Date", fontsize=12)

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax2.xaxis.set_major_locator(mdates.DayLocator(interval=1))
plt.setp(ax2.get_xticklabels(), rotation=45, ha="right")

# Shared axis formatting
for ax in (ax1, ax2):
    ax.tick_params(axis="both", labelsize=11)
    ax.set_xlim(start_dt, end_dt)
    ax.set_ylim(22, 35)
    ax.legend(loc="lower right", fontsize=10, frameon=False)
    ax.grid(True, linestyle="--", alpha=0.4)

## Case 2: May 11 - May 22, 2024 Heat Wave

In [ ]:
# Settings for plotting
start_date = "2024-05-11"
end_date = "2024-05-22"

start_dt = datetime.strptime(start_date, "%Y-%m-%d")
end_dt = datetime.strptime(end_date, "%Y-%m-%d")
month_str = start_dt.strftime("%B")

unit = r"$^\circ C$"

# Create figure and axes
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, dpi=300)
plt.subplots_adjust(hspace=0.25)

# Tmax on ax1:
ax1.plot(
    bco_obs.loc[start_date:end_date].index,
    bco_obs.loc[start_date:end_date]["tmax"],
    label="BCO Tmax",
    color="#1f77b4",
    linewidth=2
)
# ax1.plot(
#     bco_abs_daily_tmax.sel(time=slice(start_date, end_date)).time,
#     bco_abs_daily_tmax.sel(time=slice(start_date, end_date)),
#     label="BCO Tmax resampled",
#     color="#1f77b4", linestyle=":",
#     linewidth=2
# )
ax1.plot(
    tmax_obs.loc[start_date:end_date].index,
    tmax_obs.loc[start_date:end_date]["Barbados_GAIA"],
    label="GAIA Tmax",
    color="#ff7f0e",
    linewidth=2
)
ax1.plot(
    tmax_obs.loc[start_date:end_date].index,
    tmax_obs.loc[start_date:end_date]["Barbados_CIMH"],
    label="CIMH Tmax",
    color="#ee0505",
    linewidth=2
)

ax1.set_title(f"Daily Maximum Temperature — Barbados ({month_str} {start_dt.day}–{end_dt.day}, {start_dt.year})",
              fontsize=14, weight="bold", pad=12)
ax1.set_ylabel(f"Tmax / {unit}", fontsize=12)

# Tmin on ax2:
ax2.plot(
    bco_obs.loc[start_date:end_date].index,
    bco_obs.loc[start_date:end_date]["tmin"],
    label="BCO Tmin",
    color="#1f77b4",
    linewidth=2
)
# ax2.plot(
#     bco_abs_daily_tmin.sel(time=slice(start_date, end_date)).time,
#     bco_abs_daily_tmin.sel(time=slice(start_date, end_date)),
#     label="BCO Tmin resampled",
#     color="#1f77b4",
#     linewidth=2, linestyle=":"
# )
ax2.plot(
    tmin_obs.loc[start_date:end_date].index,
    tmin_obs.loc[start_date:end_date]["Barbados_GAIA"],
    label="GAIA Tmin",
    color="#ff7f0e",
    linewidth=2
)
ax2.plot(
    tmin_obs.loc[start_date:end_date].index,
    tmin_obs.loc[start_date:end_date]["Barbados_CIMH"],
    label="CIMH Tmin",
    color="#ee0505",
    linewidth=2
)

ax2.set_ylabel(f"Tmin / {unit}", fontsize=12)
ax2.set_xlabel("Date", fontsize=12)

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax2.xaxis.set_major_locator(mdates.DayLocator(interval=1))
plt.setp(ax2.get_xticklabels(), rotation=45, ha="right")

# Shared axis formatting
for ax in (ax1, ax2):
    ax.tick_params(axis="both", labelsize=11)
    ax.set_xlim(start_dt, end_dt)
    ax.set_ylim(22, 35)
    ax.legend(loc="lower right", fontsize=10, frameon=False)
    ax.grid(True, linestyle="--", alpha=0.4)

In [ ]:
# Settings for plotting
start_date = "2024-05-11"
end_date = "2024-05-22"

start_dt = datetime.strptime(start_date, "%Y-%m-%d")
end_dt = datetime.strptime(end_date, "%Y-%m-%d")
month_str = start_dt.strftime("%B")

unit = r"$^\circ C$"

# Create figure and axes
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, dpi=300)
plt.subplots_adjust(hspace=0.25)

# Tmax on ax1:
ax1.plot(
    bco_daily_tmax.sel(time=slice(start_date, end_date)).time,
    bco_daily_tmax.sel(time=slice(start_date, end_date)),
    label="BCO Tmax",
    color="#1f77b4",
    linewidth=2.5, linestyle=":"
)
ax1.plot(
    bco_abs_daily_tmax.sel(time=slice(start_date, end_date)).time,
    bco_abs_daily_tmax.sel(time=slice(start_date, end_date)),
    label="BCO Absolute Tmax",
    color="#1f77b4",
)
ax1.plot(
    tmax_obs.loc[start_date:end_date].index,
    tmax_obs.loc[start_date:end_date]["Barbados_GAIA"],
    label="GAIA Tmax",
    color="#ff7f0e",
    linewidth=2
)
ax1.plot(
    tmax_obs.loc[start_date:end_date].index,
    tmax_obs.loc[start_date:end_date]["Barbados_CIMH"],
    label="CIMH Tmax",
    color="#ee0505",
    linewidth=2
)

ax1.set_title(f"Daily Maximum Temperature — Barbados ({month_str} {start_dt.day}–{end_dt.day}, {start_dt.year})",
              fontsize=14, weight="bold", pad=12)
ax1.set_ylabel(f"Tmax / {unit}", fontsize=12)

# Tmin on ax2:
ax2.plot(
    bco_daily_tmin.sel(time=slice(start_date, end_date)).time,
    bco_daily_tmin.sel(time=slice(start_date, end_date)),
    label="BCO Tmin",
    color="#1f77b4",
    linestyle=":",
    linewidth=2.5
)
ax2.plot(
    bco_abs_daily_tmin.sel(time=slice(start_date, end_date)).time,
    bco_abs_daily_tmin.sel(time=slice(start_date, end_date)),
    label="BCO Absolute Tmin",
    color="#1f77b4",
    linewidth=2.5
)
ax2.plot(
    tmin_obs.loc[start_date:end_date].index,
    tmin_obs.loc[start_date:end_date]["Barbados_GAIA"],
    label="GAIA Tmin",
    color="#ff7f0e",
    linewidth=2
)
ax2.plot(
    tmin_obs.loc[start_date:end_date].index,
    tmin_obs.loc[start_date:end_date]["Barbados_CIMH"],
    label="CIMH Tmin",
    color="#ee0505",
    linewidth=2
)

ax2.set_ylabel(f"Tmin / {unit}", fontsize=12)
ax2.set_xlabel("Date", fontsize=12)

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax2.xaxis.set_major_locator(mdates.DayLocator(interval=1))
plt.setp(ax2.get_xticklabels(), rotation=45, ha="right")

# Shared axis formatting
for ax in (ax1, ax2):
    ax.tick_params(axis="both", labelsize=11)
    ax.set_xlim(start_dt, end_dt)
    ax.set_ylim(22, 35)
    ax.legend(loc="lower right", fontsize=10, frameon=False)
    ax.grid(True, linestyle="--", alpha=0.4)
    
plt.show()

## Problem Area 1: September 16th and 17th, 2016 

In [ ]:
# Settings for plotting
start_date = "2016-09-01"
end_date = "2016-09-30"

start_dt = datetime.strptime(start_date, "%Y-%m-%d")
end_dt = datetime.strptime(end_date, "%Y-%m-%d")
month_str = start_dt.strftime("%B")

unit = r"$^\circ C$"

# Create figure and axes
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, dpi=300)
plt.subplots_adjust(hspace=0.25)

# Tmax on ax1:
ax1.plot(
    bco_obs.loc[start_date:end_date].index,
    bco_obs.loc[start_date:end_date]["tmax"],
    label="BCO Tmax",
    color="#1f77b4",
    linewidth=2
)
# ax1.plot(
#     bco_abs_daily_tmax.sel(time=slice(start_date, end_date)).time,
#     bco_abs_daily_tmax.sel(time=slice(start_date, end_date)),
#     label="BCO Tmax resampled",
#     color="#1f77b4", linestyle=":",
#     linewidth=2
# )
ax1.plot(
    tmax_obs.loc[start_date:end_date].index,
    tmax_obs.loc[start_date:end_date]["Barbados_GAIA"],
    label="GAIA Tmax",
    color="#ff7f0e",
    linewidth=2
)
ax1.plot(
    tmax_obs.loc[start_date:end_date].index,
    tmax_obs.loc[start_date:end_date]["Barbados_CIMH"],
    label="CIMH Tmax",
    color="#ee0505",
    linewidth=2
)

ax1.set_title(f"Daily Maximum Temperature — Barbados ({month_str} {start_dt.day}–{end_dt.day}, {start_dt.year})",
              fontsize=14, weight="bold", pad=12)
ax1.set_ylabel(f"Tmax / {unit}", fontsize=12)

# Tmin on ax2:
ax2.plot(
    bco_obs.loc[start_date:end_date].index,
    bco_obs.loc[start_date:end_date]["tmin"],
    label="BCO Tmin",
    color="#1f77b4",
    linewidth=2
)
# ax2.plot(
#     bco_abs_daily_tmin.sel(time=slice(start_date, end_date)).time,
#     bco_abs_daily_tmin.sel(time=slice(start_date, end_date)),
#     label="BCO Tmin resampled",
#     color="#1f77b4",
#     linewidth=2, linestyle=":"
# )
ax2.plot(
    tmin_obs.loc[start_date:end_date].index,
    tmin_obs.loc[start_date:end_date]["Barbados_GAIA"],
    label="GAIA Tmin",
    color="#ff7f0e",
    linewidth=2
)
ax2.plot(
    tmin_obs.loc[start_date:end_date].index,
    tmin_obs.loc[start_date:end_date]["Barbados_CIMH"],
    label="CIMH Tmin",
    color="#ee0505",
    linewidth=2
)

ax2.set_ylabel(f"Tmin / {unit}", fontsize=12)
ax2.set_xlabel("Date", fontsize=12)

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax2.xaxis.set_major_locator(mdates.DayLocator(interval=1))
plt.setp(ax2.get_xticklabels(), rotation=45, ha="right")

# Shared axis formatting
for ax in (ax1, ax2):
    ax.tick_params(axis="both", labelsize=8)
    ax.set_xlim(start_dt, end_dt)
    ax.legend(loc="upper left", fontsize=10, frameon=False)
    ax.grid(True, linestyle="--", alpha=0.4)

### Problem Area 2: Late 2014

In [ ]:
# Settings for plotting
start_date = "2014-09-15"
end_date = "2014-11-25"

start_dt = datetime.strptime(start_date, "%Y-%m-%d")
end_dt = datetime.strptime(end_date, "%Y-%m-%d")
month_str = start_dt.strftime("%B")

unit = r"$^\circ C$"

# Create figure and axes
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, dpi=300)
plt.subplots_adjust(hspace=0.25)

# Tmax on ax1:
ax1.plot(
    bco_obs.loc[start_date:end_date].index,
    bco_obs.loc[start_date:end_date]["tmax"],
    label="BCO Tmax",
    color="#1f77b4",
    linewidth=2
)
# ax1.plot(
#     bco_abs_daily_tmax.sel(time=slice(start_date, end_date)).time,
#     bco_abs_daily_tmax.sel(time=slice(start_date, end_date)),
#     label="BCO Tmax resampled",
#     color="#1f77b4", linestyle=":",
#     linewidth=2
# )
ax1.plot(
    tmax_obs.loc[start_date:end_date].index,
    tmax_obs.loc[start_date:end_date]["Barbados_GAIA"],
    label="GAIA Tmax",
    color="#ff7f0e",
    linewidth=2
)
ax1.plot(
    tmax_obs.loc[start_date:end_date].index,
    tmax_obs.loc[start_date:end_date]["Barbados_CIMH"],
    label="CIMH Tmax",
    color="#ee0505",
    linewidth=2
)

ax1.set_title(f"Daily Maximum Temperature — Barbados ({month_str} {start_dt.day}–{end_dt.day}, {start_dt.year})",
              fontsize=14, weight="bold", pad=12)
ax1.set_ylabel(f"Tmax / {unit}", fontsize=12)
ax1.set_ylim(24, 35)

# Tmin on ax2:
ax2.plot(
    bco_obs.loc[start_date:end_date].index,
    bco_obs.loc[start_date:end_date]["tmin"],
    label="BCO Tmin",
    color="#1f77b4",
    linewidth=2
)
# ax2.plot(
#     bco_abs_daily_tmin.sel(time=slice(start_date, end_date)).time,
#     bco_abs_daily_tmin.sel(time=slice(start_date, end_date)),
#     label="BCO Tmin resampled",
#     color="#1f77b4",
#     linewidth=2, linestyle=":"
# )
ax2.plot(
    tmin_obs.loc[start_date:end_date].index,
    tmin_obs.loc[start_date:end_date]["Barbados_GAIA"],
    label="GAIA Tmin",
    color="#ff7f0e",
    linewidth=2
)
ax2.plot(
    tmin_obs.loc[start_date:end_date].index,
    tmin_obs.loc[start_date:end_date]["Barbados_CIMH"],
    label="CIMH Tmin",
    color="#ee0505",
    linewidth=2
)

ax2.set_ylabel(f"Tmin / {unit}", fontsize=12)
ax2.set_xlabel("Date", fontsize=12)

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax2.xaxis.set_major_locator(mdates.DayLocator(interval=1))
plt.setp(ax2.get_xticklabels(), rotation=45, ha="right")

# Shared axis formatting
for ax in (ax1, ax2):
    ax.tick_params(axis="x", labelsize=5)
    ax.set_xlim(start_dt, end_dt)
    ax.legend(loc="lower left", fontsize=10, frameon=False)
    ax.grid(True, linestyle="--", alpha=0.4)

### Problem area 3: 2013

In [ ]:
# Settings for plotting
start_date = "2013-03-15"
end_date = "2013-10-10"

start_dt = datetime.strptime(start_date, "%Y-%m-%d")
end_dt = datetime.strptime(end_date, "%Y-%m-%d")
month_str = start_dt.strftime("%B")

unit = r"$^\circ C$"

# Create figure and axes
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, dpi=300)
plt.subplots_adjust(hspace=0.25)

# Tmax on ax1:
ax1.plot(
    bco_obs.loc[start_date:end_date].index,
    bco_obs.loc[start_date:end_date]["tmax"],
    label="BCO Tmax",
    color="#1f77b4",
    linewidth=2
)
# ax1.plot(
#     bco_abs_daily_tmax.sel(time=slice(start_date, end_date)).time,
#     bco_abs_daily_tmax.sel(time=slice(start_date, end_date)),
#     label="BCO Tmax resampled",
#     color="#1f77b4", linestyle=":",
#     linewidth=2
# )
ax1.plot(
    tmax_obs.loc[start_date:end_date].index,
    tmax_obs.loc[start_date:end_date]["Barbados_GAIA"],
    label="GAIA Tmax",
    color="#ff7f0e",
    linewidth=2
)
ax1.plot(
    tmax_obs.loc[start_date:end_date].index,
    tmax_obs.loc[start_date:end_date]["Barbados_CIMH"],
    label="CIMH Tmax",
    color="#ee0505",
    linewidth=2
)

ax1.set_title(f"Daily Maximum Temperature — Barbados ({month_str} {start_dt.day}–{end_dt.day}, {start_dt.year})",
              fontsize=14, weight="bold", pad=12)
ax1.set_ylabel(f"Tmax / {unit}", fontsize=12)

# Tmin on ax2:
ax2.plot(
    bco_obs.loc[start_date:end_date].index,
    bco_obs.loc[start_date:end_date]["tmin"],
    label="BCO Tmin",
    color="#1f77b4",
    linewidth=2
)
# ax2.plot(
#     bco_abs_daily_tmin.sel(time=slice(start_date, end_date)).time,
#     bco_abs_daily_tmin.sel(time=slice(start_date, end_date)),
#     label="BCO Tmin resampled",
#     color="#1f77b4",
#     linewidth=2, linestyle=":"
# )
ax2.plot(
    tmin_obs.loc[start_date:end_date].index,
    tmin_obs.loc[start_date:end_date]["Barbados_GAIA"],
    label="GAIA Tmin",
    color="#ff7f0e",
    linewidth=2
)
ax2.plot(
    tmin_obs.loc[start_date:end_date].index,
    tmin_obs.loc[start_date:end_date]["Barbados_CIMH"],
    label="CIMH Tmin",
    color="#ee0505",
    linewidth=2
)

ax2.set_ylabel(f"Tmin / {unit}", fontsize=12)
ax2.set_xlabel("Date", fontsize=12)

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax2.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=1, interval=1))
plt.setp(ax2.get_xticklabels(), ha="right")

# Shared axis formatting
for ax in (ax1, ax2):
    ax.tick_params(axis="x", labelsize=12)
    ax.set_xlim(start_dt, end_dt)
    ax.legend(loc="lower left", fontsize=10, frameon=False)
    ax.grid(True, linestyle="--", alpha=0.4)
    
plt.show()

# Missing BCO data periods


In [ ]:
def find_missing_gaps(df: pd.DataFrame, column) -> pd.DataFrame:
    """
    Identify all contiguous missing-data gaps in a datetime-indexed dataframe.

    Parameters
    ----------
    df : pandas.DataFrame
        A dataframe with a DatetimeIndex.
    column : str
        Column name to check for missing values.

    Returns
    -------
    pandas.DataFrame
        Columns: start_date, end_date, length (days)
    """

    # --- Ensure clean datetime index ---
    df = df.copy()
    df.index = pd.to_datetime(df.index, errors="coerce")
    df = df[~df.index.isna()]
    df = df.sort_index()

    # --- Reindex to full daily range ---
    full_range = pd.date_range(df.index.min(), df.index.max(), freq="D")
    df_full = df.reindex(full_range)

    # --- Boolean Series of missing days for that column ---
    missing = df_full[column].isna()

    # --- Group where missing changes state (True→False or False→True) ---
    group_id = missing.ne(missing.shift()).cumsum()

    # --- Summarize each contiguous block ---
    summary = (
        missing.groupby(group_id)
        .agg(start=lambda s: s.index[0],
             end=lambda s: s.index[-1],
             val=lambda s: s.iloc[0],
             length=lambda s: s.size)
    )

    # --- Keep only missing blocks ---
    gaps = summary[summary["val"]].drop(columns="val")
    gaps = gaps.rename(columns={"start": "start_date", "end": "end_date"})
    gaps = gaps.reset_index(drop=True)

    return gaps

find_missing_gaps(bco_obs, "tmax")

## Find Data Gaps

In [ ]:
def find_missing_gaps(df: pd.DataFrame, columns: list =None) -> pd.DataFrame:
    """
    Identify gaps in daily time series data where specified columns are missing.

    Parameters:
    - df: pandas DataFrame with a datetime index or a 'Date' column
    - columns: list of column names to check for missing data (default: all columns)

    Returns:
    - gap_df: DataFrame with start_date, end_date, and gap_length_days for each missing block
    """
    # --- 0. Ensure datetime index ---
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        if "Date" in df.columns:
            df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
            df = df.set_index("Date")
        else:
            raise ValueError("DataFrame must have a datetime index or a 'Date' column.")
    df.index = pd.to_datetime(df.index, errors="coerce")
    df = df[~df.index.isna()].sort_index()
    df = df[~df.index.duplicated(keep="first")]

    # --- 1. Reindex to complete daily range ---
    full_range = pd.date_range(df.index.min(), df.index.max(), freq="D")
    df_full = df.reindex(full_range)

    # --- 2. Define missing rows ---
    if columns is None:
        is_missing = df_full.isnull().any(axis=1)
    else:
        is_missing = df_full[columns].isnull().any(axis=1)

    # --- 3. Group and summarize contiguous missing blocks ---
    group_id = is_missing.ne(is_missing.shift()).cumsum()
    summary = (
        is_missing
        .groupby(group_id)
        .agg(start=lambda s: s.index[0],
             end=lambda s: s.index[-1],
             is_missing=lambda s: s.iloc[0],
             length=lambda s: s.size)
        .reset_index(drop=True)
    )

    # --- 4. Extract only missing blocks ---
    gap_df = summary.loc[summary["is_missing"], ["start", "end", "length"]].rename(
        columns={"start": "start_date", "end": "end_date", "length": "gap_length_days"}
    ).reset_index(drop=True)

    return gap_df

In [ ]:
find_missing_gaps(bco_obs, ["tmax"])

# Comparing BCO with GAIA and CIMH
## Scatter Plots

In [ ]:
unit = r"$^\circ$C"

# Filter spurious points
bco_tmax_clean = bco_obs[(bco_obs["tmax"] >= 20) & (bco_obs["tmax"] <= 40)]
bco_tmin_clean = bco_obs[(bco_obs["tmin"] >= 15) & (bco_obs["tmin"] <= 35)]

# Align Tmax datasets
df_rain = pd.DataFrame({
    "BCO": bco_tmax_clean["tmax"],
    "GAIA": tmax_obs["Barbados_GAIA"],
    "CIMH": tmax_obs["Barbados_CIMH"]
}).dropna()

# Align Tmin datasets
df_tmin = pd.DataFrame({
    "BCO": bco_tmin_clean["tmin"],
    "GAIA": tmin_obs["Barbados_GAIA"],
    "CIMH": tmin_obs["Barbados_CIMH"]
}).dropna()

# Initialize figure
fig, axes = plt.subplots(2, 2, figsize=(16, 14), dpi=300)
ax1, ax2, ax3, ax4 = axes.flat

# ============================================================
#  AX1 — BCO vs GAIA Tmax
# ============================================================
x = df_rain["BCO"]
y = df_rain["GAIA"]

ax1.scatter(x, y, alpha=0.5, s=10)

lim_min = min(x.min(), y.min())
lim_max = max(x.max(), y.max())
ax1.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=1)

m, c = np.polyfit(x, y, 1)
ax1.plot([lim_min, lim_max], m*np.array([lim_min, lim_max]) + c, 'k-', lw=1)

corr = x.corr(y)
r2 = corr**2
rmse = np.sqrt(((y - x)**2).mean())
mae = np.abs(y - x).mean()
bias = (y - x).mean()

stats1 = (
    f"y = {m:.2f}x + {c:.2f}\n"
    f"Corr = {corr:.2f}\n"
    f"R² = {r2:.2f}\n"
    f"RMSE = {rmse:.2f}\n"
    f"MAE = {mae:.2f}\n"
    f"Bias = {bias:.2f}"
)
ax1.text(0.03, 0.97, stats1, transform=ax1.transAxes, va="top",
         bbox=dict(facecolor="white", alpha=0.8), fontsize=15)

ax1.set_title("BCO vs GAIA Tmax")
ax1.set_xlabel(f"BCO Tmax / {unit}")
ax1.set_ylabel(f"GAIA Tmax / {unit}")
ax1.set_xlim(24, 36)
ax1.set_ylim(24, 36)
ax1.grid(True)

# ============================================================
#  AX2 — BCO vs CIMH Tmax
# ============================================================
x = df_rain["BCO"]
y = df_rain["CIMH"]

ax2.scatter(x, y, alpha=0.5, s=10)

lim_min = min(x.min(), y.min())
lim_max = max(x.max(), y.max())
ax2.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=1)

m, c = np.polyfit(x, y, 1)
ax2.plot([lim_min, lim_max], m*np.array([lim_min, lim_max]) + c, 'k-', lw=1)

corr = x.corr(y)
r2 = corr**2
rmse = np.sqrt(((y - x)**2).mean())
mae = np.abs(y - x).mean()
bias = (y - x).mean()

stats2 = (
    f"y = {m:.2f}x + {c:.2f}\n"
    f"Corr = {corr:.2f}\n"
    f"R² = {r2:.2f}\n"
    f"RMSE = {rmse:.2f}\n"
    f"MAE = {mae:.2f}\n"
    f"Bias = {bias:.2f}"
)
ax2.text(0.03, 0.97, stats2, transform=ax2.transAxes, va="top",
         bbox=dict(facecolor="white", alpha=0.8), fontsize=15)

ax2.set_title("BCO vs CIMH Tmax")
ax2.set_xlabel(f"BCO Tmax / {unit}")
ax2.set_ylabel(f"CIMH Tmax / {unit}")
ax2.set_xlim(24, 36)
ax2.set_ylim(24, 36)
ax2.grid(True)

# ============================================================
#  AX3 — BCO vs GAIA Tmin
# ============================================================
x = df_tmin["BCO"]
y = df_tmin["GAIA"]

ax3.scatter(x, y, alpha=0.5, s=10)

lim_min = 15
lim_max = 33
ax3.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=1)

m, c = np.polyfit(x, y, 1)
ax3.plot([lim_min, lim_max], m*np.array([lim_min, lim_max]) + c, 'k-', lw=1)

corr = x.corr(y)
r2 = corr**2
rmse = np.sqrt(((y - x)**2).mean())
mae = np.abs(y - x).mean()
bias = (y - x).mean()

stats3 = (
    f"y = {m:.2f}x + {c:.2f}\n"
    f"Corr = {corr:.2f}\n"
    f"R² = {r2:.2f}\n"
    f"RMSE = {rmse:.2f}\n"
    f"MAE = {mae:.2f}\n"
    f"Bias = {bias:.2f}"
)
ax3.text(0.03, 0.97, stats3, transform=ax3.transAxes, va="top",
         bbox=dict(facecolor="white", alpha=0.8), fontsize=15)

ax3.set_title("BCO vs GAIA Tmin")
ax3.set_xlabel(f"BCO Tmin / {unit}")
ax3.set_ylabel(f"GAIA Tmin / {unit}")
ax3.set_xlim(15, 33)
ax3.set_ylim(15, 33)
ax3.grid(True)

# ============================================================
#  AX4 — BCO vs CIMH Tmin
# ============================================================
x = df_tmin["BCO"]
y = df_tmin["CIMH"]

ax4.scatter(x, y, alpha=0.5, s=10)

lim_min = min(x.min(), y.min())
lim_max = max(x.max(), y.max())
ax4.plot([15, 33], [15, 33], 'r--', lw=1)

m, c = np.polyfit(x, y, 1)
ax4.plot([15, 33], m*np.array([15, 33]) + c, 'k-', lw=1)

corr = x.corr(y)
r2 = corr**2
rmse = np.sqrt(((y - x)**2).mean())
mae = np.abs(y - x).mean()
bias = (y - x).mean()

stats4 = (
    f"y = {m:.2f}x + {c:.2f}\n"
    f"Corr = {corr:.2f}\n"
    f"R² = {r2:.2f}\n"
    f"RMSE = {rmse:.2f}\n"
    f"MAE = {mae:.2f}\n"
    f"Bias = {bias:.2f}"
)
ax4.text(0.03, 0.97, stats4, transform=ax4.transAxes, va="top",
         bbox=dict(facecolor="white", alpha=0.8), fontsize=15)

ax4.set_title("BCO vs CIMH Tmin")
ax4.set_xlabel(f"BCO Tmin / {unit}")
ax4.set_ylabel(f"CIMH Tmin / {unit}")
ax4.set_xlim(15, 33)
ax4.set_ylim(15, 33)
ax4.grid(True)

fig.suptitle("BCO vs CIMH vs GAIA Scatterplots and Stats", fontsize=16, weight="bold", y=0.95)
# ============================================================
# FINAL
# ============================================================
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(fname="BCO vs CIMH vs GAIA Scatterplots and Stats.png", dpi=300)
plt.show()

In [ ]:
unit = r"$^\circ$C"

# ============================================================
# CLEAN FILTERS
# ============================================================
bco_tmax_clean = bco_obs[(bco_obs["tmax"] >= 20) & (bco_obs["tmax"] <= 40)]
bco_tmin_clean = bco_obs[(bco_obs["tmin"] >= 15) & (bco_obs["tmin"] <= 35)]

# ============================================================
# ALIGN DATAFRAMES (NOW INCLUDING WIND)
# ============================================================
df_rain = pd.DataFrame({
    "BCO": bco_tmax_clean["tmax"],
    "GAIA": tmax_obs["Barbados_GAIA"],
    "CIMH": tmax_obs["Barbados_CIMH"],
    "wind": bco_tmax_clean["vmean"],
}).dropna()

df_tmin = pd.DataFrame({
    "BCO": bco_tmin_clean["tmin"],
    "GAIA": tmin_obs["Barbados_GAIA"],
    "CIMH": tmin_obs["Barbados_CIMH"],
    "wind": bco_tmin_clean["vmean"],
}).dropna()

from matplotlib.colors import BoundaryNorm, ListedColormap

# ============================================================
# WIND BINNING: discrete steps every 2 m/s from 0–12
# ============================================================
bounds = np.arange(0, 12, 2)      # 0,2,4,6,8,10,12
n_colors = len(bounds) - 1
colors = [
    "#b3e5fc",  # light blue (0–2)
    "#4fc3f7",  # blue (2–4)
    "#81c784",  # green (4–6)
    "#fff176",  # yellow (6–8)
    "#ffb74d",  # orange (8–10)
    "#e57373",  # red (10–12)
]

cmap = ListedColormap(colors)
norm = BoundaryNorm(bounds, cmap.N)

# ============================================================
# CREATE FIGURE
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 14), dpi=300)
ax1, ax2, ax3, ax4 = axes.flat

# ============================================================
# AX1 — BCO vs GAIA Tmax
# ============================================================
x = df_rain["BCO"]
y = df_rain["GAIA"]
sc1 = ax1.scatter(x, y, c=df_rain["wind"], cmap=cmap, norm=norm,
                  alpha=0.7, s=12)

lim_min = min(x.min(), y.min())
lim_max = max(x.max(), y.max())
ax1.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=1)

m, c = np.polyfit(x, y, 1)
ax1.plot([lim_min, lim_max], m*np.array([lim_min, lim_max]) + c, 'k-', lw=1)

corr = x.corr(y)
r2 = corr**2
rmse = np.sqrt(((y - x)**2).mean())
mae = np.abs(y - x).mean()
bias = (y - x).mean()

stats1 = (
    f"y = {m:.2f}x + {c:.2f}\n"
    f"Corr = {corr:.2f}\n"
    f"R² = {r2:.2f}\n"
    f"RMSE = {rmse:.2f}\n"
    f"MAE = {mae:.2f}\n"
    f"Bias = {bias:.2f}"
)
ax1.text(0.03, 0.97, stats1, transform=ax1.transAxes, va="top",
         bbox=dict(facecolor="white", alpha=0.8), fontsize=15)

ax1.set_title("BCO vs GAIA Tmax", fontsize=15)
ax1.set_xlabel(f"BCO Tmax / {unit}", fontsize=15)
ax1.set_ylabel(f"GAIA Tmax / {unit}", fontsize=15)
ax1.set_xlim(24, 36)
ax1.set_ylim(24, 36)
ax1.grid(True)

# ============================================================
# AX2 — BCO vs CIMH Tmax
# ============================================================
x = df_rain["BCO"]
y = df_rain["CIMH"]

sc2 = ax2.scatter(x, y, c=df_rain["wind"], cmap=cmap, norm=norm,
                  alpha=0.7, s=12)

lim_min = min(x.min(), y.min())
lim_max = max(x.max(), y.max())
ax2.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=1)

m, c = np.polyfit(x, y, 1)
ax2.plot([lim_min, lim_max], m*np.array([lim_min, lim_max]) + c, 'k-', lw=1)

corr = x.corr(y)
r2 = corr**2
rmse = np.sqrt(((y - x)**2).mean())
mae = np.abs(y - x).mean()
bias = (y - x).mean()

stats2 = (
    f"y = {m:.2f}x + {c:.2f}\n"
    f"Corr = {corr:.2f}\n"
    f"R² = {r2:.2f}\n"
    f"RMSE = {rmse:.2f}\n"
    f"MAE = {mae:.2f}\n"
    f"Bias = {bias:.2f}"
)
ax2.text(0.03, 0.97, stats2, transform=ax2.transAxes, va="top",
         bbox=dict(facecolor="white", alpha=0.8), fontsize=15)

ax2.set_title("BCO vs CIMH Tmax", fontsize=15)
ax2.set_xlabel(f"BCO Tmax / {unit}", fontsize=15)
ax2.set_ylabel(f"CIMH Tmax / {unit}", fontsize=15)
ax2.set_xlim(24, 36)
ax2.set_ylim(24, 36)
ax2.grid(True)

# ============================================================
# AX3 — BCO vs GAIA Tmin
# ============================================================
x = df_tmin["BCO"]
y = df_tmin["GAIA"]

sc3 = ax3.scatter(x, y, c=df_tmin["wind"], cmap=cmap, norm=norm,
                  alpha=0.7, s=12)

ax3.plot([15, 33], [15, 33], 'r--', lw=1)

m, c = np.polyfit(x, y, 1)
ax3.plot([15, 33], m*np.array([15, 33]) + c, 'k-', lw=1)

corr = x.corr(y)
r2 = corr**2
rmse = np.sqrt(((y - x)**2).mean())
mae = np.abs(y - x).mean()
bias = (y - x).mean()

stats3 = (
    f"y = {m:.2f}x + {c:.2f}\n"
    f"Corr = {corr:.2f}\n"
    f"R² = {r2:.2f}\n"
    f"RMSE = {rmse:.2f}\n"
    f"MAE = {mae:.2f}\n"
    f"Bias = {bias:.2f}"
)
ax3.text(0.03, 0.97, stats3, transform=ax3.transAxes, va="top",
         bbox=dict(facecolor="white", alpha=0.8), fontsize=15)

ax3.set_title("BCO vs GAIA Tmin", fontsize=15)
ax3.set_xlabel(f"BCO Tmin / {unit}", fontsize=15)
ax3.set_ylabel(f"GAIA Tmin / {unit}", fontsize=15)
ax3.set_xlim(15, 33)
ax3.set_ylim(15, 33)
ax3.grid(True)

# ============================================================
# AX4 — BCO vs CIMH Tmin
# ============================================================
x = df_tmin["BCO"]
y = df_tmin["CIMH"]

sc4 = ax4.scatter(x, y, c=df_tmin["wind"], cmap=cmap, norm=norm,
                  alpha=0.7, s=12)

ax4.plot([15, 33], [15, 33], 'r--', lw=1)

m, c = np.polyfit(x, y, 1)
ax4.plot([15, 33], m*np.array([15, 33]) + c, 'k-', lw=1)

corr = x.corr(y)
r2 = corr**2
rmse = np.sqrt(((y - x)**2).mean())
mae = np.abs(y - x).mean()
bias = (y - x).mean()

stats4 = (
    f"y = {m:.2f}x + {c:.2f}\n"
    f"Corr = {corr:.2f}\n"
    f"R² = {r2:.2f}\n"
    f"RMSE = {rmse:.2f}\n"
    f"MAE = {mae:.2f}\n"
    f"Bias = {bias:.2f}"
)
ax4.text(0.03, 0.97, stats4, transform=ax4.transAxes, va="top",
         bbox=dict(facecolor="white", alpha=0.8), fontsize=15)

ax4.set_title("BCO vs CIMH Tmin", fontsize=15)
ax4.set_xlabel(f"BCO Tmin / {unit}", fontsize=15)
ax4.set_ylabel(f"CIMH Tmin / {unit}", fontsize=15)
ax4.set_xlim(15, 33)
ax4.set_ylim(15, 33)
ax4.grid(True)

# ============================================================
# HORIZONTAL COLORBAR (extra bottom padding to avoid overlap)
# ============================================================

# Reserve clean bottom space for the colorbar
fig.subplots_adjust(bottom=0.15)

# Create a fixed-position colorbar axis at bottom center
cbar_ax = fig.add_axes([0.15, 0.06, 0.70, 0.035])  # [left, bottom, width, height]

cbar = fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap=cmap),
    cax=cbar_ax,
    orientation="horizontal"
)

cbar.set_label(r"Wind speed / m s$^{-1}$", fontsize=15)
cbar.set_ticks(bounds)

# ============================================================
# FINAL LAYOUT
# ============================================================
fig.suptitle("BCO vs CIMH vs GAIA Scatterplots and Stats", fontsize=20, weight="bold", y=0.98)

plt.tight_layout(rect=[0, 0.12, 1, 0.96])
plt.savefig("BCO vs CIMH vs GAIA Scatterplots and Stats.png", dpi=300)
plt.show()


In [ ]:
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

unit = r"$^\circ$C"

# ============================================================
# CLEAN FILTERS
# ============================================================
bco_tmax_clean = bco_obs[(bco_obs["tmax"] >= 20) & (bco_obs["tmax"] <= 40)]
bco_tmin_clean = bco_obs[(bco_obs["tmin"] >= 15) & (bco_obs["tmin"] <= 35)]

# ============================================================
# ALIGN DATAFRAMES (USING WIND DIRECTION)
# ============================================================
df_rain = pd.DataFrame({
    "BCO": bco_tmax_clean["tmax"],
    "GAIA": tmax_obs["Barbados_GAIA"],
    "CIMH": tmax_obs["Barbados_CIMH"],
    "wind": bco_tmax_clean["dir_mean"],   # degrees
}).dropna()

df_tmin = pd.DataFrame({
    "BCO": bco_tmin_clean["tmin"],
    "GAIA": tmin_obs["Barbados_GAIA"],
    "CIMH": tmin_obs["Barbados_CIMH"],
    "wind": bco_tmin_clean["dir_mean"],   # degrees
}).dropna()

# ============================================================
# DIVERGENT COLORMAP CENTERED AT 90°
# ============================================================
vmin = 50
vcenter = 90
vmax = 130

cmap = plt.cm.RdBu_r
norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

# ============================================================
# CREATE FIGURE
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 14), dpi=300)
ax1, ax2, ax3, ax4 = axes.flat

# ============================================================
# AX1 — BCO vs GAIA Tmax
# ============================================================
x = df_rain["BCO"]
y = df_rain["GAIA"]

ax1.scatter(x, y, c=df_rain["wind"], cmap=cmap, norm=norm,
            alpha=0.7, s=12)

lim_min = min(x.min(), y.min())
lim_max = max(x.max(), y.max())
ax1.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=1)

m, c = np.polyfit(x, y, 1)
ax1.plot([lim_min, lim_max],
         m * np.array([lim_min, lim_max]) + c, 'k-', lw=1)

corr = x.corr(y)
ax1.text(
    0.03, 0.97,
    f"y = {m:.2f}x + {c:.2f}\n"
    f"Corr = {corr:.2f}\n"
    f"R² = {corr**2:.2f}\n"
    f"RMSE = {np.sqrt(((y - x)**2).mean()):.2f}\n"
    f"MAE = {np.abs(y - x).mean():.2f}\n"
    f"Bias = {(y - x).mean():.2f}",
    transform=ax1.transAxes, va="top",
    bbox=dict(facecolor="white", alpha=0.8), fontsize=15
)

ax1.set_title("BCO vs GAIA Tmax", fontsize=15)
ax1.set_xlabel(f"BCO Tmax / {unit}", fontsize=15)
ax1.set_ylabel(f"GAIA Tmax / {unit}", fontsize=15)
ax1.set_xlim(24, 36)
ax1.set_ylim(24, 36)
ax1.grid(True)

# ============================================================
# AX2 — BCO vs CIMH Tmax
# ============================================================
x = df_rain["BCO"]
y = df_rain["CIMH"]

ax2.scatter(x, y, c=df_rain["wind"], cmap=cmap, norm=norm,
            alpha=0.7, s=12)

ax2.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=1)

m, c = np.polyfit(x, y, 1)
ax2.plot([lim_min, lim_max],
         m * np.array([lim_min, lim_max]) + c, 'k-', lw=1)

corr = x.corr(y)
ax2.text(
    0.03, 0.97,
    f"y = {m:.2f}x + {c:.2f}\n"
    f"Corr = {corr:.2f}\n"
    f"R² = {corr**2:.2f}\n"
    f"RMSE = {np.sqrt(((y - x)**2).mean()):.2f}\n"
    f"MAE = {np.abs(y - x).mean():.2f}\n"
    f"Bias = {(y - x).mean():.2f}",
    transform=ax2.transAxes, va="top",
    bbox=dict(facecolor="white", alpha=0.8), fontsize=15
)

ax2.set_title("BCO vs CIMH Tmax", fontsize=15)
ax2.set_xlabel(f"BCO Tmax / {unit}", fontsize=15)
ax2.set_ylabel(f"CIMH Tmax / {unit}", fontsize=15)
ax2.set_xlim(24, 36)
ax2.set_ylim(24, 36)
ax2.grid(True)

# ============================================================
# AX3 — BCO vs GAIA Tmin
# ============================================================
x = df_tmin["BCO"]
y = df_tmin["GAIA"]

ax3.scatter(x, y, c=df_tmin["wind"], cmap=cmap, norm=norm,
            alpha=0.7, s=12)

ax3.plot([15, 33], [15, 33], 'r--', lw=1)

m, c = np.polyfit(x, y, 1)
ax3.plot([15, 33],
         m * np.array([15, 33]) + c, 'k-', lw=1)

corr = x.corr(y)
ax3.text(
    0.03, 0.97,
    f"y = {m:.2f}x + {c:.2f}\n"
    f"Corr = {corr:.2f}\n"
    f"R² = {corr**2:.2f}\n"
    f"RMSE = {np.sqrt(((y - x)**2).mean()):.2f}\n"
    f"MAE = {np.abs(y - x).mean():.2f}\n"
    f"Bias = {(y - x).mean():.2f}",
    transform=ax3.transAxes, va="top",
    bbox=dict(facecolor="white", alpha=0.8), fontsize=15
)

ax3.set_title("BCO vs GAIA Tmin", fontsize=15)
ax3.set_xlabel(f"BCO Tmin / {unit}", fontsize=15)
ax3.set_ylabel(f"GAIA Tmin / {unit}", fontsize=15)
ax3.set_xlim(15, 33)
ax3.set_ylim(15, 33)
ax3.grid(True)

# ============================================================
# AX4 — BCO vs CIMH Tmin
# ============================================================
x = df_tmin["BCO"]
y = df_tmin["CIMH"]

ax4.scatter(x, y, c=df_tmin["wind"], cmap=cmap, norm=norm,
            alpha=0.7, s=12)

ax4.plot([15, 33], [15, 33], 'r--', lw=1)

m, c = np.polyfit(x, y, 1)
ax4.plot([15, 33],
         m * np.array([15, 33]) + c, 'k-', lw=1)

corr = x.corr(y)
ax4.text(
    0.03, 0.97,
    f"y = {m:.2f}x + {c:.2f}\n"
    f"Corr = {corr:.2f}\n"
    f"R² = {corr**2:.2f}\n"
    f"RMSE = {np.sqrt(((y - x)**2).mean()):.2f}\n"
    f"MAE = {np.abs(y - x).mean():.2f}\n"
    f"Bias = {(y - x).mean():.2f}",
    transform=ax4.transAxes, va="top",
    bbox=dict(facecolor="white", alpha=0.8), fontsize=15
)

ax4.set_title("BCO vs CIMH Tmin", fontsize=15)
ax4.set_xlabel(f"BCO Tmin / {unit}", fontsize=15)
ax4.set_ylabel(f"CIMH Tmin / {unit}", fontsize=15)
ax4.set_xlim(15, 33)
ax4.set_ylim(15, 33)
ax4.grid(True)

# ============================================================
# COLORBAR (FIXED)
# ============================================================
fig.subplots_adjust(bottom=0.15)
cbar_ax = fig.add_axes([0.15, 0.06, 0.70, 0.035])

cbar = fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap=cmap),
    cax=cbar_ax,
    orientation="horizontal"
)

cbar.set_label(r"Wind direction / $^\circ$", fontsize=15)
cbar.set_ticks([50, 70, 90, 110, 130])

# Center marker at 90° (axes coordinates, NOT data coordinates)
cbar.ax.plot([0.5, 0.5], [0, 1],
             transform=cbar.ax.transAxes,
             color="k", lw=1)

# ============================================================
# FINAL LAYOUT
# ============================================================
fig.suptitle(
    "BCO vs CIMH vs GAIA Scatterplots and Stats",
    fontsize=20, weight="bold", y=0.98
)

plt.tight_layout(rect=[0, 0.12, 1, 0.96])
plt.savefig("BCO vs CIMH vs GAIA Temperature Scatte.png", dpi=300)
plt.show()


In [ ]:
from scipy.stats import gaussian_kde

unit = "mm"

# ============================================================
# ALIGN DATAFRAMES (NOW INCLUDING WIND)
# ============================================================
df_rain = pd.DataFrame({
    "BCO": bco_obs["total_rainfall"],
    "GAIA": rain_obs["Barbados_GAIA"],
    "CIMH": rain_obs["Barbados_CIMH"],
    "wind": bco_obs["vmean"],
}).dropna()

# ============================================================
# WIND BINNING: discrete steps every 2 m/s from 0–12
# ============================================================
# bounds = np.arange(0, 12, 2)      # 0,2,4,6,8,10,12
# n_colors = len(bounds) - 1
# colors = [
#     "#b3e5fc",  # light blue (0–2)
#     "#4fc3f7",  # blue (2–4)
#     "#81c784",  # green (4–6)
#     "#fff176",  # yellow (6–8)
#     "#ffb74d",  # orange (8–10)
#     "#e57373",  # red (10–12)
# ]

# cmap = ListedColormap(colors)
# norm = BoundaryNorm(bounds, cmap.N)

# ============================================================
# CREATE FIGURE
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(8, 10), dpi=300)
ax1, ax2 = axes.flat

# ============================================================
# AX1 — BCO vs GAIA Tmax
# ============================================================
x = df_rain["BCO"]
y = df_rain["GAIA"]
sc1 = ax1.scatter(x, y, alpha=0.7, s=12) # c=df_rain["wind"], cmap=cmap, norm=norm,

lim_min = min(x.min(), y.min())
lim_max = max(x.max(), y.max())
ax1.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=1)

m, c = np.polyfit(x, y, 1)
ax1.plot([lim_min, lim_max], m*np.array([lim_min, lim_max]) + c, 'k-', lw=1)

corr = x.corr(y)
r2 = corr**2
rmse = np.sqrt(((y - x)**2).mean())
mae = np.abs(y - x).mean()
bias = (y - x).mean()

stats1 = (
    f"y = {m:.2f}x + {c:.2f}\n"
    f"Corr = {corr:.2f}\n"
    f"R² = {r2:.2f}\n"
    f"RMSE = {rmse:.2f}\n"
    f"MAE = {mae:.2f}\n"
    f"Bias = {bias:.2f}"
)
ax1.text(0.80, 0.37, stats1, transform=ax1.transAxes, va="top",
         bbox=dict(facecolor="white", alpha=0.8), fontsize=11)

ax1.set_title("BCO vs GAIA daily rainfall", fontsize=15)
ax1.set_xlabel(f"BCO daily rainfall / {unit}", fontsize=15)
ax1.set_ylabel(f"GAIA daily rainfall / {unit}", fontsize=15)
ax1.set_xlim(0, 80)
ax1.set_ylim(0, 80)
ax1.grid(True)

# ============================================================
# AX2 — BCO vs CIMH daily rainfall
# ============================================================
x = df_rain["BCO"]
y = df_rain["CIMH"]

sc2 = ax2.scatter(x, y, alpha=0.7, s=12) # c=df_rain["wind"], cmap=cmap, norm=norm,

lim_min = min(x.min(), y.min())
lim_max = max(x.max(), y.max())
ax2.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=1)

m, c = np.polyfit(x, y, 1)
ax2.plot([lim_min, lim_max], m*np.array([lim_min, lim_max]) + c, 'k-', lw=1)

corr = x.corr(y)
r2 = corr**2
rmse = np.sqrt(((y - x)**2).mean())
mae = np.abs(y - x).mean()
bias = (y - x).mean()

stats2 = (
    f"y = {m:.2f}x + {c:.2f}\n"
    f"Corr = {corr:.2f}\n"
    f"R² = {r2:.2f}\n"
    f"RMSE = {rmse:.2f}\n"
    f"MAE = {mae:.2f}\n"
    f"Bias = {bias:.2f}"
)
ax2.text(0.80, 0.37, stats2, transform=ax2.transAxes, va="top",
         bbox=dict(facecolor="white", alpha=0.8), fontsize=11)

ax2.set_title("BCO vs CIMH daily rainfall", fontsize=15)
ax2.set_xlabel(f"BCO daily rainfall / {unit}", fontsize=15)
ax2.set_ylabel(f"CIMH daily rainfall / {unit}", fontsize=15)
ax2.set_xlim(0, 80)
ax2.set_ylim(0, 80)
ax2.grid(True)

# ============================================================
# HORIZONTAL COLORBAR (extra bottom padding to avoid overlap)
# ============================================================

# Reserve clean bottom space for the colorbar
fig.subplots_adjust(bottom=0.15)

# Create a fixed-position colorbar axis at bottom center
cbar_ax = fig.add_axes([0.15, 0.06, 0.70, 0.02])  # [left, bottom, width, height]

cbar = fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap=cmap),
    cax=cbar_ax,
    orientation="horizontal"
)

cbar.set_label(r"Wind speed / m s$^{-1}$", fontsize=15)
cbar.set_ticks(bounds)

# ============================================================
# FINAL LAYOUT
# ============================================================
fig.suptitle("BCO vs CIMH vs GAIA daily rainfall scatterplots and Stats", fontsize=16, weight="bold", y=0.98)

plt.tight_layout(rect=[0, 0.12, 1, 0.96])
plt.savefig("BCO vs CIMH vs GAIA Scatterplots and Stats (rain).png", dpi=300)
plt.show()

In [ ]:
from scipy.stats import gaussian_kde

unit = "mm"

# ============================================================
# ALIGN DATAFRAMES (NOW INCLUDING WIND)
# ============================================================
df_rain = pd.DataFrame({
    "BCO": bco_obs["total_rainfall"],
    "GAIA": rain_obs["Barbados_GAIA"],
    "CIMH": rain_obs["Barbados_CIMH"],
    "wind": bco_obs["vmean"],
}).dropna()

# ============================================================
# CREATE FIGURE
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(8, 10), dpi=300)
ax1, ax2 = axes.flat

# ============================================================
# AX1 — BCO vs GAIA daily rainfall
# ============================================================
x = df_rain["BCO"].values
y = df_rain["GAIA"].values

xy = np.vstack([x, y])
density = gaussian_kde(xy)(xy)
density_norm = (density - density.min()) / (density.max() - density.min())
alpha_vals = 0.4 + 0.6 * density_norm

idx = density.argsort()
x, y, alpha_vals = x[idx], y[idx], alpha_vals[idx]

sc1 = ax1.scatter(x, y, alpha=alpha_vals, s=12, edgecolors="none", rasterized=True)

lim_min = min(x.min(), y.min())
lim_max = max(x.max(), y.max())
ax1.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=1)

m, c = np.polyfit(x, y, 1)
ax1.plot([lim_min, lim_max], m*np.array([lim_min, lim_max]) + c, 'k-', lw=1)

corr = np.corrcoef(x, y)[0, 1]
r2 = corr**2
rmse = np.sqrt(((y - x)**2).mean())
mae = np.abs(y - x).mean()
bias = (y - x).mean()

stats1 = (
    f"y = {m:.2f}x + {c:.2f}\n"
    f"Corr = {corr:.2f}\n"
    f"R² = {r2:.2f}\n"
    f"RMSE = {rmse:.2f}\n"
    f"MAE = {mae:.2f}\n"
    f"Bias = {bias:.2f}"
)

ax1.text(0.80, 0.37, stats1, transform=ax1.transAxes, va="top",
         bbox=dict(facecolor="white", alpha=0.8), fontsize=11)

ax1.set_title("BCO vs GAIA daily rainfall", fontsize=15)
ax1.set_xlabel(f"BCO daily rainfall / {unit}", fontsize=15)
ax1.set_ylabel(f"GAIA daily rainfall / {unit}", fontsize=15)
ax1.set_xlim(0, 80)
ax1.set_ylim(0, 80)
ax1.grid(True)

# ============================================================
# AX2 — BCO vs CIMH daily rainfall
# ============================================================
x = df_rain["BCO"].values
y = df_rain["CIMH"].values

xy = np.vstack([x, y])
density = gaussian_kde(xy)(xy)
density_norm = (density - density.min()) / (density.max() - density.min())
alpha_vals = 0.4 + 0.6 * density_norm

idx = density.argsort()
x, y, alpha_vals = x[idx], y[idx], alpha_vals[idx]

sc2 = ax2.scatter(x, y, alpha=alpha_vals, s=12, edgecolors="none", rasterized=True)

lim_min = min(x.min(), y.min())
lim_max = max(x.max(), y.max())
ax2.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=1)

m, c = np.polyfit(x, y, 1)
ax2.plot([lim_min, lim_max], m*np.array([lim_min, lim_max]) + c, 'k-', lw=1)

corr = np.corrcoef(x, y)[0, 1]
r2 = corr**2
rmse = np.sqrt(((y - x)**2).mean())
mae = np.abs(y - x).mean()
bias = (y - x).mean()

stats2 = (
    f"y = {m:.2f}x + {c:.2f}\n"
    f"Corr = {corr:.2f}\n"
    f"R² = {r2:.2f}\n"
    f"RMSE = {rmse:.2f}\n"
    f"MAE = {mae:.2f}\n"
    f"Bias = {bias:.2f}"
)

ax2.text(0.80, 0.37, stats2, transform=ax2.transAxes, va="top",
         bbox=dict(facecolor="white", alpha=0.8), fontsize=11)

ax2.set_title("BCO vs CIMH daily rainfall", fontsize=15)
ax2.set_xlabel(f"BCO daily rainfall / {unit}", fontsize=15)
ax2.set_ylabel(f"CIMH daily rainfall / {unit}", fontsize=15)
ax2.set_xlim(0, 80)
ax2.set_ylim(0, 80)
ax2.grid(True)

# ============================================================
# FINAL LAYOUT
# ============================================================
fig.suptitle(
    "BCO vs CIMH vs GAIA daily rainfall scatterplots and Stats",
    fontsize=16, weight="bold", y=0.98
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("BCO vs CIMH vs GAIA Scatterplots and Stats (rain).png", dpi=300)
plt.show()

## Scatter plots of Tmean
We also plot the mean temperatures from the datasets. The stations compute mean daily temperatures by taking the average value of the Tmax and the Tmin datasets. However, the BCO dataset has sub-hourly observations which means that it is possible to compute a more accurate mean daily temperature. We will do both and compare them with the mean daily temperatures from the BCO and the GAIA. 

### BCO and GAIA Tmean

In [ ]:
tmean_obs = ((tmax_obs + tmin_obs) / 2).round(1)
tmean_obs.to_csv("1985-2024 - Daily Tmean.csv", index_label="Date")

In [ ]:
tmean_obs

### BCO Tmean data

# Loading Windspeed Data

In [ ]:
from datetime import datetime, timedelta
import intake

cat = intake.open_catalog("https://tcodata.mpimet.mpg.de/catalog.yaml")
wxt = cat.BCO.surfacemet_wxt_v1.to_dask()

In [ ]:
wxt

In [ ]:
from datetime import datetime, timedelta
import intake

cat = intake.open_catalog("https://tcodata.mpimet.mpg.de/catalog.yaml")
wxt = cat.BCO.surfacemet_wxt_v1.to_dask()
wxt["VEL"].sel(time=slice(datetime.now() - timedelta(days=30), datetime.now())).plot(figsize=(12, 4));